# Destek 00 — Veri tanıları ve panel veri varsayım testleri

**Amaç:** `countries_clean.csv` panel verisini analizlerden önce istatistiksel olarak tanımak: veri bütünlüğü, durağanlık, yatay kesit bağımlılığı, otokorelasyon, değişen varyans. Sonuç: g01 / g05 / g06 için hangi yöntem uygun, hangisi değil.

**Girdi:** `data/clean/countries_clean.csv` (237 ülke × 1950–2100; 2101 dışarıda). Gözlem dönemi 1950–2023 (T=74), 2024+ BM projeksiyonu.

**Önemli not:** g01, g05, g06 *betimleyici toplulaştırma* işleridir (pay, oran, sayım). Bu notebook'taki testler bir tahmin modeli kurmak için değil, verinin yapısını ve hangi çıkarım yöntemlerinin güvenilmez olacağını belgelemek içindir.

In [9]:
%pip install linearmodels
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from linearmodels.panel import PanelOLS

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
raw = pd.read_csv(ROOT / "data/clean/countries_clean.csv")
df  = raw[raw.Time <= 2100].copy()          # 2101 dışarıda (yalnızca 1 Ocak nüfusu var)
obs = df[df.Time <= 2023].copy()            # gözlem dönemi
obs["log_pop"] = np.log(obs.TPopulation1July / 1000)   # milyon → log
print(f"Tüm veri: {raw.shape}, 2101 hariç: {df.shape}, gözlem dönemi: {obs.shape}")

Note: you may need to restart the kernel to use updated packages.
Tüm veri: (36024, 68), 2101 hariç: (35787, 68), gözlem dönemi: (17538, 69)


## 1. Veri bütünlüğü

In [10]:
key = ["TPopulation1July","MedianAgePop","NatChange","NatChangeRT","PopChange","CNMR","NetMigrations","TFR","LEx"]
big = df[df.Time==2024].loc[lambda s: s.TPopulation1July > 1000, "LocID"]   # büyük ülkeler (nüfus bin kişi)
x = df.dropna(subset=["BirthsMale"])
r = (df.NatChange + df.NetMigrations - df.PopChange).abs()
w = df.groupby("Time").TPopulation1July.sum() / 1e6

kontrol = pd.DataFrame([
 ("Ülke sayısı = 237",                            df.LocID.nunique(), 237),
 ("Satır = 237 × 151 yıl (1950–2100)",            len(df), 237*151),
 ("Tekrarlanan (LocID, Time)",                    int(df.duplicated(["LocID","Time"]).sum()), 0),
 ("Ana değişkenlerde eksik (2101 hariç)",         int(df[key].isna().sum().sum()), 0),
 ("İlk projeksiyon yılı",                         int(df[df.IsProjection].Time.min()), 2024),
 ("Erkek+Kadın doğum ≠ Doğum (>0.01 bin)",        int(((x.BirthsMale + x.BirthsFemale - x.Births).abs() > 0.01).sum()), 0),
 ("NatChange+NetMigrations ≠ PopChange (>5 bin)", int((r > 5).sum()), 0),
 ("Nüfusu >1 milyon ülke sayısı (2024)",          int(len(big)), 161),
 ("Dünya nüfusu zirve yılı (kontrol #1)",         int(w.idxmax()), 2084),
 ("Dünya zirve nüfusu (milyar, 2 hane)",          round(w.max(), 2), 10.29),
], columns=["kontrol","bulunan","beklenen"])
kontrol["tamam"] = kontrol.bulunan == kontrol.beklenen
display(kontrol)
ulk = ROOT / "data/clean/ulkeler_1m.csv"
if ulk.exists(): print("ulkeler_1m.csv satır sayısı:", len(pd.read_csv(ulk)))
print("BirthsMale/Female eksik olan ülke:", df[df.BirthsMale.isna()].Location.unique())
print("TFR'si sabit olan ülke:", obs.groupby("Location").TFR.std().loc[lambda s: s<1e-8].index.tolist())

,kontrol,bulunan,beklenen,tamam
0,Ülke sayısı = 237,237.00,237.00,True
1,Satır = 237 × 151 yıl (1950–2100),35787.00,35787.00,True
2,"Tekrarlanan (LocID, Time)",0.00,0.00,True
3,Ana değişkenlerde eksik (2101 hariç),0.00,0.00,True
4,İlk projeksiyon yılı,2024.00,2024.00,True
5,Erkek+Kadın doğum ≠ Doğum (>0.01 bin),0.00,0.00,True
6,NatChange+NetMigrations ≠ PopChange (>5 bin),0.00,0.00,True
7,Nüfusu >1 milyon ülke sayısı (2024),161.00,161.00,True
8,Dünya nüfusu zirve yılı (kontrol #1),2084.00,2084.00,True
9,"Dünya zirve nüfusu (milyar, 2 hane)",10.29,10.29,True


ulkeler_1m.csv satır sayısı: 161
BirthsMale/Female eksik olan ülke: ['Holy See']
TFR'si sabit olan ülke: ['Holy See']


**Okuma:** Dünya satırı bu dosyada yok; "ülke toplamı = Dünya satırı" kontrolü ham BM dosyasındaki Dünya satırıyla yapılmalı (burada yalnızca zirve yılı ve değeri doğrulanıyor). Vatikan'ın TFR'si sabit (1.0), panel testlerinde dışarıda bırakılır.

## 2. Verinin doğası ve model seçimi

- **Panel yapısı:** N=237 ülke, T=74 yıl, dengeli panel. Uzun panel sayılır; makro panel yöntemleri (CIPS, Driscoll–Kraay) uygun.
- **Değişkenler BM modeli çıktısı:** 1950–2023 tahminî ve yumuşatılmış, 2024+ tamamen projeksiyon. Rastgele gözlem değil, kohort-bileşen modeli çıktısı; bu yüzden "hata terimi" yorumu sınırlı.
- **Beklenti:** güçlü trend (durağan değil), ülkeler arası güçlü ortak hareket (demografik geçiş), yüksek otokorelasyon, büyük–küçük ülke arasında çok farklı varyans.

Aşağıdaki testler bu beklentileri sınar.

## 3. Durağanlık (panel birim kök)

In [11]:
vars_ = ["log_pop", "TFR", "LEx", "MedianAgePop"]
obs_t = obs[obs.Location != "Holy See"]
W = {v: obs_t.pivot(index="Time", columns="LocID", values=v) for v in vars_}

def adf_fisher(M, reg="ct", diff=False):
    ps = []
    for col in M.columns:
        y = M[col].values
        if diff: y = np.diff(y)
        ps.append(adfuller(y, regression=reg, autolag="AIC")[1])
    ps = np.clip(np.array(ps), 1e-12, 1)
    stat = -2*np.log(ps).sum(); N = len(ps)
    return (ps < .05).mean(), stat, stats.chi2.sf(stat, 2*N)

def kpss_rej(M):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")     # KPSS tablo aralığı uyarıları (p-değeri sınırda)
        return np.mean([kpss(M[c].values, regression="ct", nlags="auto")[1] < .05 for c in M.columns])

rows = []
for v in vars_:
    a, s, p = adf_fisher(W[v]);  ad, sd, pd_ = adf_fisher(W[v], "c", True)
    rows.append((v, round(a,3), round(p,4), round(ad,3), round(kpss_rej(W[v]),3)))
bt = pd.DataFrame(rows, columns=["değişken","ADF red oranı (seviye, trend)","Maddala-Wu p (seviye)","ADF red oranı (1. fark)","KPSS red oranı (H0: durağan)"])
display(bt)

,değişken,"ADF red oranı (seviye, trend)",Maddala-Wu p (seviye),ADF red oranı (1. fark),KPSS red oranı (H0: durağan)
0,log_pop,0.102,0.0855,0.131,0.907
1,TFR,0.110,0.0000,0.403,0.881
2,LEx,0.208,0.0000,0.555,0.835
3,MedianAgePop,0.182,0.0000,0.136,0.953


**Okuma:** Ülke bazında ADF, birim kökü seviyede çoğu ülke için reddedemiyor (düşük red oranı); KPSS ise durağanlığı ülkelerin büyük çoğunluğunda reddediyor. İkisi aynı yönde: **seviyeler durağan değil.** Maddala–Wu (Fisher) birleşik p-değeri ülkeler arası bağımsızlık gerektirir; aşağıdaki yatay kesit bağımlılığı çok güçlü olduğundan bu p-değerlerine güvenilmez. Bu yüzden ortak faktöre dayanıklı **Pesaran CIPS** testine bakıyoruz.

In [12]:
def cadf_t(Y, p=1):
    T, N = Y.shape; yb = Y.mean(1); dyb = np.diff(yb); ts = []
    for i in range(N):
        y = Y[:, i]; dy = np.diff(y); rows = np.arange(p, T-1); dep = dy[rows]
        X = [np.ones(len(rows)), (rows+1.), y[rows], yb[rows], dyb[rows]]
        for k in range(1, p+1): X += [dy[rows-k], dyb[rows-k]]
        X = np.column_stack(X); b = np.linalg.lstsq(X, dep, rcond=None)[0]
        e = dep - X@b; s2 = e@e/(len(dep)-X.shape[1]); cov = s2*np.linalg.inv(X.T@X)
        ts.append(b[2]/np.sqrt(cov[2,2]))
    return np.array(ts)
cips = lambda Y, p=1: np.clip(cadf_t(Y, p), -6.19, 6.19).mean()

# H0 (birim kök + ortak faktör) altında simülasyonla yaklaşık kritik değer
rng = np.random.default_rng(42); T0, N0 = W["TFR"].shape
sim = []
for _ in range(500):
    f = rng.normal(size=T0); g = rng.uniform(.5, 1.5, N0)
    Y = np.cumsum(np.outer(f, g) + rng.normal(size=(T0, N0)), axis=0); sim.append(cips(Y))
cv = np.quantile(sim, [.01, .05, .10])
res = pd.DataFrame({"değişken": vars_, "CIPS": [round(cips(W[v].values), 3) for v in vars_]})
res["%5 kritik (yaklaşık)"] = round(cv[1], 3)
res["birim kök reddedildi (%5)"] = res.CIPS < cv[1]
display(res); print("Simüle kritik değerler (%1, %5, %10):", cv.round(3))

,değişken,CIPS,%5 kritik (yaklaşık),birim kök reddedildi (%5)
0,log_pop,-3.475,-2.535,True
1,TFR,-2.495,-2.535,False
2,LEx,-2.475,-2.535,False
3,MedianAgePop,-3.863,-2.535,True


Simüle kritik değerler (%1, %5, %10): [-2.611 -2.535 -2.482]


**Okuma:** Ortak faktör kontrol edilince sonuç değişkene göre değişiyor: nüfus (log) ve medyan yaş için birim kök reddediliyor; **TFR ve yaşam beklentisi için reddedilemiyor.** Yani durağanlık kanıtı karışık. Bu, BM serilerinin pürüzsüz, trendli yapısıyla uyumlu. Pratik sonuç: seviyeler üzerinde klasik regresyon çıkarımı yapmayın (sahte regresyon riski); pay, oran, fark ve betimleyici karşılaştırma kullanın. (CIPS kritik değerleri burada simülasyonla yaklaşık bulundu; Pesaran 2007 tablosuyla tam aynı olmayabilir.)

## 4. Yatay kesit bağımlılığı (Pesaran CD)

In [13]:
def pesaran_cd(M):
    M = M.dropna(axis=1); T, N = M.shape
    r = np.corrcoef(M.values.T)[np.triu_indices(N, 1)]
    return N, T, np.sqrt(2*T/(N*(N-1)))*r.sum(), r.mean(), np.abs(r).mean()

def detrend(M):
    t = np.arange(len(M)); return M.apply(lambda s: s - np.polyval(np.polyfit(t, s.values, 1), t))

rows = []
for v in vars_:
    for ad, M in [("1. fark", W[v].diff().dropna()), ("trendden arındırılmış", detrend(W[v]))]:
        N, T, cd, mr, mar = pesaran_cd(M); rows.append((v, ad, N, T, round(cd,1), round(mr,3), round(mar,3)))
display(pd.DataFrame(rows, columns=["değişken","seri","N","T","CD istatistiği","ort. korelasyon","ort. mutlak korelasyon"]))

,değişken,seri,N,T,CD istatistiği,ort. korelasyon,ort. mutlak korelasyon
0,log_pop,1. fark,236,73,216.7,0.152,0.359
1,log_pop,trendden arındırılmış,236,74,297.4,0.208,0.598
2,TFR,1. fark,236,73,156.8,0.110,0.231
3,TFR,trendden arındırılmış,236,74,182.8,0.128,0.507
4,LEx,1. fark,236,73,212.4,0.149,0.197
5,LEx,trendden arındırılmış,236,74,271.8,0.190,0.456
6,MedianAgePop,1. fark,236,73,519.2,0.365,0.412
7,MedianAgePop,trendden arındırılmış,236,74,890.8,0.622,0.682


**Okuma:** CD istatistikleri yüzlerce (kritik değer ≈ 1.96), ortalama çiftli korelasyon pozitif. **Yatay kesit bağımlılığı çok güçlü**: ülkeler ortak demografik geçiş sürecinden geçiyor ve BM aynı yöntemi hepsine uyguluyor. Sonuçlar: (i) ülkeleri bağımsız gözlem sayan standart hatalar yanıltıcı, (ii) "237 bağımsız ülke" gibi cümleler kurulmamalı, (iii) birinci kuşak panel birim kök testleri (IPS, LLC, Fisher) geçersiz, CIPS gibi ikinci kuşak test gerekir.

## 5. Referans model: otokorelasyon ve değişen varyans

Tanı amacıyla basit bir ilişki: **TFR_it = α_i + β·LEx_it + ε_it** (demografik geçişin temel ilişkisi). Amaç β'yı yorumlamak değil, hata yapısını görmek.

In [14]:
P = obs_t.set_index(["LocID", "Time"]); y = P["TFR"]; X = P[["LEx"]]
fe = PanelOLS(y, X, entity_effects=True).fit()
print(f"FE (ülke etkili): β_LEx = {fe.params['LEx']:.4f}, within-R² = {fe.rsquared_within:.3f}")
print(f"Ülke etkileri gerekli mi (F)? F = {fe.f_pooled.stat:.1f}, p = {fe.f_pooled.pval:.3g}")

# Mundlak (Hausman'a dayanıklı alternatif): ülke ortalamasının katsayısı 0 mı?
Z = P[["TFR", "LEx"]].copy(); Z["LEx_ort"] = Z.groupby(level=0).LEx.transform("mean")
mm = sm.OLS(Z.TFR, sm.add_constant(Z[["LEx", "LEx_ort"]])).fit(cov_type="cluster", cov_kwds={"groups": Z.index.get_level_values(0)})
print(f"Mundlak: ülke ortalaması katsayısı p = {mm.pvalues['LEx_ort']:.3g} →", "sabit etkiler (FE) tercih edilir" if mm.pvalues['LEx_ort'] < .05 else "rastgele etki kabul edilebilir")

# Otokorelasyon: Wooldridge (2002), 1. fark hataları için H0: ρ = -0.5
d = P[["TFR", "LEx"]].groupby(level=0).diff().dropna()
u = sm.OLS(d.TFR, d[["LEx"]]).fit().resid.to_frame("u"); u["ul"] = u.groupby(level=0).u.shift(); u = u.dropna()
m2 = sm.OLS(u.u, u[["ul"]]).fit(cov_type="cluster", cov_kwds={"groups": u.index.get_level_values(0)})
tw = (m2.params["ul"] + .5) / m2.bse["ul"]
print(f"Wooldridge: ρ̂ = {m2.params['ul']:.3f} (H0: -0.5), F = {tw**2:.1f}, p = {2*stats.norm.sf(abs(tw)):.3g}")
Rm = fe.resids.unstack(0)
ar1 = [np.corrcoef(Rm[c].values[1:], Rm[c].values[:-1])[0, 1] for c in Rm.columns]
print(f"FE artıklarının ülke ortalaması AR(1) katsayısı = {np.nanmean(ar1):.3f}")

# Değişen varyans: gruplar arası değiştirilmiş Wald (Greene) ve nüfus büyüklüğüne karşı BP benzeri
e = fe.resids.to_frame("e"); e["e2"] = e.e**2
s2i = e.groupby(level=0).e2.mean(); s2 = e.e2.mean(); T = e.groupby(level=0).size().iloc[0]
Vi = (e.e2 - e.groupby(level=0).e2.transform("mean")).pow(2).groupby(level=0).sum() / (T*(T-1))
Wald = ((s2i - s2)**2 / Vi).sum(); Ng = len(s2i)
print(f"Değiştirilmiş Wald (gruplar arası değişen varyans): chi2({Ng}) = {Wald:.0f}, p = {stats.chi2.sf(Wald, Ng):.3g}")
bp = sm.OLS(e.e2.values, sm.add_constant(P.loc[e.index, "log_pop"].values)).fit()
print(f"Artık varyansı ~ log nüfus (BP benzeri): p = {bp.f_pvalue:.3g}")

# Artıklarda yatay kesit bağımlılığı
N, T_, cd, mr, mar = pesaran_cd(Rm)
print(f"Artıklarda Pesaran CD = {cd:.1f}, ortalama çiftli korelasyon = {mr:.3f}")

FE (ülke etkili): β_LEx = -0.1319, within-R² = 0.620
Ülke etkileri gerekli mi (F)? F = 65.9, p = 0
Mundlak: ülke ortalaması katsayısı p = 0.0184 → sabit etkiler (FE) tercih edilir
Wooldridge: ρ̂ = 0.491 (H0: -0.5), F = 271.0, p = 6.98e-61
FE artıklarının ülke ortalaması AR(1) katsayısı = 0.957
Değiştirilmiş Wald (gruplar arası değişen varyans): chi2(236) = 123018, p = 0
Artık varyansı ~ log nüfus (BP benzeri): p = 4.12e-24
Artıklarda Pesaran CD = 171.7, ortalama çiftli korelasyon = 0.120


### Standart hata kıyaslaması: varsayım ihlalinin çıkarıma etkisi

In [15]:
se = {}
se["Klasik (homoskedastik, bağımsız)"]              = PanelOLS(y, X, entity_effects=True).fit(cov_type="unadjusted")
se["Ülke kümeli (otokorelasyon + değişen varyans)"] = PanelOLS(y, X, entity_effects=True).fit(cov_type="clustered", cluster_entity=True)
se["Driscoll-Kraay (+ yatay kesit bağımlılığı)"]    = PanelOLS(y, X, entity_effects=True).fit(cov_type="kernel")
tab = pd.DataFrame({k: {"beta": r.params["LEx"], "SH": r.std_errors["LEx"], "t": r.tstats["LEx"]} for k, r in se.items()}).T.round(4)
display(tab)
tw_fit = PanelOLS(y, X, entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
print(f"Ülke + yıl sabit etkili model: beta = {tw_fit.params['LEx']:.4f} (yıl etkisi eklenince katsayı belirgin değişiyor: ortak zaman şokları var)")

,beta,SH,t
"Klasik (homoskedastik, bağımsız)",-0.1319,0.0008,-167.7046
Ülke kümeli (otokorelasyon + değişen varyans),-0.1319,0.0045,-29.1686
Driscoll-Kraay (+ yatay kesit bağımlılığı),-0.1319,0.0073,-18.0081


Ülke + yıl sabit etkili model: beta = -0.0607 (yıl etkisi eklenince katsayı belirgin değişiyor: ortak zaman şokları var)


**Okuma:** Güçlü pozitif otokorelasyon, ülkeler arasında çok farklı hata varyansı ve güçlü yatay kesit bağımlılığı birlikte var. Klasik standart hata gerçek belirsizliği kat kat küçük gösteriyor; Driscoll–Kraay ile t-istatistiği belirgin düşüyor. Yıl etkileri eklenince katsayının değişmesi ortak zaman şoklarını doğruluyor.

## 6. Gözlem–projeksiyon sınırı

In [16]:
rows = []
for v in ["TFR", "LEx", "MedianAgePop"]:
    d1 = df.pivot(index="Time", columns="LocID", values=v).diff().abs().mean(axis=1)
    rows.append((v, d1.loc[1951:2023].mean(), d1.loc[2025:2100].mean()))
display(pd.DataFrame(rows, columns=["değişken", "ort. |yıllık değişim| gözlem (1951–2023)", "ort. |yıllık değişim| proj. (2025–2100)"]).round(4))

,değişken,ort. |yıllık değişim| gözlem (1951–2023),ort. |yıllık değişim| proj. (2025–2100)
0,TFR,0.0696,0.0099
1,LEx,0.5469,0.1375
2,MedianAgePop,0.2263,0.2014


**Okuma:** TFR ve yaşam beklentisinde projeksiyon dönemi gözlem dönemine göre çok daha pürüzsüz (yıllık değişim TFR'de yaklaşık 7, yaşam beklentisinde yaklaşık 4 kat küçük; medyan yaşta fark küçük). Projeksiyon rastgele bir süreçten gelen gözlem değil, BM'nin varsayımlarının çıktısı. Bu yüzden 2024+ verisinde otokorelasyon, durağanlık ya da anlamlılık testi yorumlanamaz. Yol haritasındaki kuralla (kümeleme ve ikiz analizinde yalnızca 1950–2023) tutarlı.

## 7. Sonuç ve g01 / g05 / g06 için uygun yöntem

| Soru | Bulgu | Sonuç |
|---|---|---|
| Durağanlık | Seviyeler durağan değil (ADF/KPSS aynı yönde; CIPS TFR ve yaşam beklentisi için birim kökü reddetmiyor) | Seviyelerde regresyon çıkarımı yapma; pay/oran/değişim kullan |
| Yatay kesit bağımlılığı | Çok güçlü (CD ≫ 1.96) | Ülkeleri bağımsız örnek sayma; Fisher/IPS yerine CIPS; SH için Driscoll–Kraay |
| Otokorelasyon | Güçlü pozitif (Wooldridge reddediyor, artık AR(1) ≈ 0.96) | Kümeli ya da Driscoll–Kraay SH; klasik SH kullanma |
| Değişen varyans | Ülkeler arasında güçlü | Robust SH; büyük ülke = küçük ülke varsayma |
| Gözlem/projeksiyon | Projeksiyon pürüzsüz ve varsayım çıktısı | Test yalnızca 1950–2023; 2024+ "gözlem" değil |

**g01 (bölge payları), g05 (sağkalım), g06 (göç kalkanı)** nüfus ağırlıklı toplulaştırma, oran ve sayımdır; parametre ya da p-değeri üretmezler. Bu yüzden yukarıdaki varsayım ihlalleri bu üç görselin sayılarını bozmaz. Etkilediği yerler: (a) "237 bağımsız ülke" gibi çıkarım iddiaları, (b) yıl yıl karşılaştırmada "anlamlı fark" dili. Sunumda betimleyici dil kullanın ("pay %9'dan %37,5'e yükseliyor"), "istatistiksel olarak anlamlı" demeyin.

Bir model gerekirse (örneğin doğurganlık–yaşam beklentisi ilişkisinin nicel özeti): ülke sabit etkili panel, ülke kümeli ya da Driscoll–Kraay standart hatalı, yalnızca 1950–2023, gerekirse yıl etkili ve birinci farklarla. Bu proje için gerekli değil.